In [10]:
import pandas as pd

df = pd.read_csv("Dataset1.csv")

df = df.sort_values(["Train_No", "SN"])

df["Arrival_time"] = pd.to_datetime(df["Arrival_time"], format="%H:%M:%S")
df["Departure_Time"] = pd.to_datetime(df["Departure_Time"], format="%H:%M:%S")


journey = df.groupby("Train_No").agg(
    Start_Station=("Station_Name","first"),
    End_Station=("Station_Name","last"),
    Start_Time=("Departure_Time","first"),
    End_Time=("Arrival_time","last")
).reset_index()


# Handle next day journey
journey.loc[
    journey["End_Time"] < journey["Start_Time"],
    "End_Time"
] += pd.Timedelta(days=1)


journey["Journey_Duration_Hours"] = (
    journey["End_Time"] - journey["Start_Time"]
).dt.total_seconds()/3600


journey["Route"] = (
    journey["Start_Station"] 
    + " to " 
    + journey["End_Station"]
)


result = journey[
    ["Train_No","Route","Journey_Duration_Hours"]
]


result = result.sort_values(
    "Journey_Duration_Hours",
    ascending=False
)

result["Journey_Duration_Hours"] = result["Journey_Duration_Hours"].round(2)


      Train_No                         Route  Journey_Duration_Hours
374      11103     JHANSI JN to BANDRA TERMI                   23.92
1723     15001      MUZAFFARPUR to DEHRA DUN                   23.92
1300     12949      PORBANDAR to SANTRAGACHI                   23.92
888      12494      HAZRAT NIZAM to PUNE JN.                   23.83
1140     12778        KOCHUVELI to HUBLI JN.                   23.83
252       8617         RANCHI to ANAND VIHAR                   23.83
1128     12766          AMRAVATI to TIRUPATI                   23.83
406      11404   CHHATRAPATI to NAGPUR JN.(C                   23.75
1139     12777        HUBLI JN. to KOCHUVELI                   23.75
2074     16534  KRANTIVIRA S to BHAGAT-KI-KO                   23.75


In [13]:
# Convert decimal hours into hours and minutes format

hours = journey["Journey_Duration_Hours"].astype(int)

minutes = (
    (journey["Journey_Duration_Hours"] - hours) * 60
).round().astype(int)


journey["Journey_Duration"] = (
    hours.astype(str) 
    + " hours " 
    + minutes.astype(str) 
    + " minutes"
)


result = journey[
    ["Train_No", "Route", "Journey_Duration"]
]


result = result.sort_values(
    "Journey_Duration",
    ascending=False
)


print(result.head(10))

      Train_No                         Route    Journey_Duration
7482     58537   KORAPUT JN. to VISAKHAPATNA  9 hours 55 minutes
6732     55540    HAJIPUR JN. to KATIHAR JN.  9 hours 55 minutes
3002     22962                 HAPA to SURAT  9 hours 55 minutes
6660     55326    KASGANJ JN. to LUCKNOW JN.  9 hours 55 minutes
1926     15769    ALIPUR DUAR to LUMDING JN.  9 hours 55 minutes
2851     22692  HAZRAT NIZAM to KRANTIVIRA S  9 hours 55 minutes
1322     12973       INDORE BG to JAIPUR JN.  9 hours 55 minutes
747      12343       SEALDAH to NEW JALPAIGU  9 hours 55 minutes
2942     22889              DIGHA FS to PURI  9 hours 55 minutes
7299     57542          NANDED to MANMAD JN.  9 hours 55 minutes


In [14]:
# Save output file

result.to_csv("Route_Duration_Comparison.csv", index=False)